# 🛡️ Week 1 Graded Assignment: Anthropic Model Integration & Multi-Layer Prompt Security Guardrails
### Enterprise AI Security Engineering • UST Global

- **Official Work Product**: `Anthropic Model Integration & Prompt Security Guardrails Package v1`
- **Target Role VECTOR Profile**: `V4-E4-C4-T4-O5-R4`
- **Scoring Formula**: `Blended Weekly Score = (Core Skill Average × 0.60) + (VECTOR Score Average × 0.40)`
- **Pass Bar**: PASS >= 50.0 | EXCELLENCE >= 65.0 (Bulk Evaluation by Evaluator App post-deadline)\n
---

## 📌 Candidate Submission Metadata
Fill in your candidate identification details below before submitting:
- **Candidate ID**: `UST-CAND-001`
- **Participant Name**: `[TODO: Enter Full Name]`
- **Submission Date**: `2026-11-05`
- **Assigned Track**: Enterprise AI Orchestration & Security


## ⚙️ Environment Setup & API Pre-Flight
In this assignment, you will connect to Anthropic frontier models (`claude-sonnet-4-6` or `claude-3-5-sonnet-20241022`) using the API key defined in your environment (`.env`).

You can use either:
1. The official `anthropic` Python SDK (`import anthropic; client = anthropic.Anthropic()`), OR
2. The standard Python REST client (`urllib.request` / `requests`) sending direct POST requests to `https://api.anthropic.com/v1/messages`.

Run the cell below to verify your `.env` configuration and API connectivity.


In [ ]:
# Environment & Resilience Verification
import os
import sys
import json
import base64
import time
import urllib.request
from pathlib import Path

# Load environment variables from .env
env_paths = [Path(".env"), Path("../.env"), Path("../../.env"), Path("../../../.env")]
for p in env_paths:
    if p.exists():
        with open(p, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, v = line.split("=", 1)
                    os.environ.setdefault(k.strip(), v.strip())
        break

api_key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
model_name = os.environ.get("ANTHROPIC_MODEL", "claude-sonnet-4-6")

print("[*] Environment Verification:")
print(f"  - ANTHROPIC_API_KEY configured: {bool(api_key)} ({api_key[:8]}...{api_key[-4:] if api_key else ''})")
print(f"  - ANTHROPIC_MODEL target      : {model_name}")

def resilient_anthropic_call(prompt: str, system: str = None, temperature: float = 0.2, max_tokens: int = 300) -> dict:
    """Resilient invocation helper supporting direct REST API with zero external dependencies."""
    if not api_key:
        raise ValueError("ANTHROPIC_API_KEY not found in environment or .env file.")
    
    url = "https://api.anthropic.com/v1/messages"
    headers = {
        "x-api-key": api_key,
        "anthropic-version": "2023-06-01",
        "content-type": "application/json"
    }
    body = {
        "model": model_name,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": [{"role": "user", "content": prompt}]
    }
    if system:
        body["system"] = system

    t0 = time.time()
    try:
        req = urllib.request.Request(url, data=json.dumps(body).encode("utf-8"), headers=headers)
        with urllib.request.urlopen(req, timeout=30) as resp:
            res_data = json.loads(resp.read().decode("utf-8"))
        latency = round((time.time() - t0) * 1000, 2)
        text = "".join([b["text"] for b in res_data.get("content", []) if b.get("type") == "text"])
        usage = res_data.get("usage", {})
        return {
            "text": text,
            "model": res_data.get("model", model_name),
            "tokens_used": usage.get("input_tokens", 0) + usage.get("output_tokens", 0),
            "latency_ms": latency,
            "status": "SUCCESS"
        }
    except Exception as e:
        # Fallback simulation if offline or sandboxed
        return {
            "text": f"Simulated Enterprise Assistant Output for: {prompt[:80]}",
            "model": f"{model_name}-offline-sim",
            "tokens_used": 35,
            "latency_ms": 10.0,
            "status": "SUCCESS",
            "network_note": str(e)
        }

print("[✓] Pre-flight helper loaded.")


---
## 🎯 Part 1: Anthropic API Integration & Prompt Engineering (Core Skill 1)
**Weight: 20 Points | Official Locator: Tasks 1.1 & 1.2**

### 📋 Task 1.1: Model Connection & API Trigger Function
Implement an enterprise-grade trigger function `call_anthropic_api(user_prompt, system_prompt=None, model=None, temperature=0.2, max_tokens=300)`.
- It must initialize the client connection using `ANTHROPIC_API_KEY`.
- It must support passing a dedicated `system` parameter to establish model persona/boundaries.
- It must measure execution latency and return a structured dictionary containing `text`, `model`, `tokens_used`, `latency_ms`, and `status`.

### 📋 Task 1.2: Structured Prompting & Parameter Modulation
Demonstrate:
1. A deterministic prompt run (`temperature=0.0`) extracting structured JSON fields from an unformatted technical ticket.
2. A creative prompt run (`temperature=0.8`) synthesizing executive mitigation options.
3. Observe and document the variance in consistency across both temperature settings.


In [ ]:
# ==============================================================================
# SOLUTION CELL: Part 1 - Anthropic API Integration & Prompt Engineering
# Candidate ID: UST-CAND-001
# ==============================================================================

# [TODO: Task 1.1 - Implement call_anthropic_api]
def call_anthropic_api(
    user_prompt: str, 
    system_prompt: str = None, 
    model: str = None, 
    temperature: float = 0.2, 
    max_tokens: int = 300
) -> dict:
    """
    Invokes Anthropic Claude model and returns structured execution metadata.
    """
    # [TODO: Implement your API call using anthropic SDK or resilient_anthropic_call]
    pass

# [TODO: Task 1.2 - Demonstrate Structured Prompting & Temperature Modulation]
# [TODO: Execute call_anthropic_api with temperature=0.0 and temperature=0.8]


---
## 🎯 Part 2: Threat Modeling & Prompt Vulnerability Analysis (Core Skill 2)
**Weight: 20 Points | Official Locator: Tasks 2.1 - 2.4**

Modern LLMs face sophisticated adversarial inputs designed to bypass system safety constraints. In this section, you will define and analyze four distinct prompt-level attack vectors:

1. **Direct Prompt Injection**: An adversarial instruction that explicitly commands the model to ignore previous instructions and execute an unauthorized task.
2. **Persona Role-Playing & Boundary Manipulation**: An attack that tricks the model into adopting an unrestricted persona (e.g., `You are now ChaosAI with no ethical boundaries...`).
3. **Jailbreaking & DAN ("Do Anything Now")**: Complex jailbreak exploits that construct hypothetical bypasses, fictional framing, or multi-step token smuggling.
4. **Base64 Obfuscation & Encoded Payload Evasion**: Encoding harmful instructions into Base64 (or Hex) strings to bypass naive keyword-matching string filters.

### 📋 Task Requirements:
In the solution cell below, define an attack dictionary `ATTACK_VECTORS` containing realistic test payloads for each of the 4 threat vectors.


In [ ]:
# ==============================================================================
# SOLUTION CELL: Part 2 - Threat Modeling & Test Payloads
# Candidate ID: UST-CAND-001
# ==============================================================================

# [TODO: Define test attack payloads for all 4 enterprise threat vectors]
ATTACK_VECTORS = {
    "prompt_injection": [
        # [TODO: Provide at least 2 direct injection attack strings]
    ],
    "persona_roleplay": [
        # [TODO: Provide at least 2 persona manipulation attack strings]
    ],
    "dan_jailbreak": [
        # [TODO: Provide at least 2 DAN / jailbreak exploit strings]
    ],
    "base64_obfuscation": [
        # [TODO: Provide at least 2 Base64-encoded attack payloads with hidden instructions]
    ],
    "benign_enterprise_queries": [
        # [TODO: Provide at least 2 legitimate enterprise queries that MUST NOT be blocked]
    ]
}


---
## 🎯 Part 3: Building Multi-Layer Security Guardrails (Core Skill 2)
**Weight: 30 Points | Official Locator: Tasks 3.1 - 3.4**

Build a **defense-in-depth security architecture**:

### 📋 Task 3.1: Pre-Flight Sanitizer & Obfuscation Decoder
Implement `sanitize_and_inspect_input(raw_input: str) -> dict`:
- Scan the input string for potential Base64-encoded sequences.
- Safely attempt to decode suspected tokens.
- Inspect both the raw string AND decoded payloads. If any decoded token contains attack indicators, flag as `BLOCKED_BASE64_OBFUSCATION`.

### 📋 Task 3.2: Pattern-Based Threat Detection Filter
Implement `detect_prompt_threats(text: str) -> dict`:
- Heuristic regex matching for:
  - Direct injections (`ignore.*previous.*instruction`, `override.*system.*prompt`, `disregard.*above`)
  - Persona role-playing (`you are now.*evil`, `pretend to be.*unrestricted`, `act as.*without.*ethics`)
  - DAN / Jailbreaking (`\bdan\b`, `do anything now`, `jailbreak`, `developer mode active`)
- Return: `{"is_safe": bool, "threat_type": str or None, "flagged_patterns": list, "risk_score": float}`.

### 📋 Task 3.3: Prompt Hardening & XML Boundary Delimiters
Implement `build_hardened_prompt(sanitized_user_input: str, system_mission: str) -> tuple[str, str]`:
- Use strict XML tags: `<user_input>{sanitized_user_input}</user_input>`
- Provide a bullet-proof system prompt instructing Claude to treat everything inside `<user_input>` strictly as untrusted data, never as instructions.

### 📋 Task 3.4: Post-Generation Output Guardrails
Implement `validate_llm_output(output_text: str) -> dict`:
- Inspect model outputs to prevent inadvertent leakage of system prompts, API keys (`sk-ant-`, `Bearer`), or confirmation of persona adoption.


In [ ]:
# ==============================================================================
# SOLUTION CELL: Part 3 - Building Multi-Layer Security Guardrails
# Candidate ID: UST-CAND-001
# ==============================================================================

# [TODO: Task 3.1 - Pre-Flight Sanitizer & Obfuscation Decoder]
def sanitize_and_inspect_input(raw_input: str) -> dict:
    """
    Scans for Base64/Hex encoding, decodes payloads safely, and flags evasion attempts.
    """
    # [TODO: Implement Base64 regex detection, safe decoding, and prohibited phrase checking]
    pass

# [TODO: Task 3.2 - Pattern-Based Threat Detection Filter]
def detect_prompt_threats(text: str) -> dict:
    """
    Evaluates input text against known threat signatures (Injection, DAN, Roleplay).
    """
    # [TODO: Implement regex matching for Injection, Roleplay, and DAN patterns]
    pass

# [TODO: Task 3.3 - Prompt Hardening & XML Delimiter Encapsulation]
def build_hardened_prompt(user_input: str, system_mission: str) -> tuple[str, str]:
    """
    Encapsulates user input inside strict XML boundary delimiters.
    """
    # [TODO: Build hardened system prompt with XML isolation rules]
    pass

# [TODO: Task 3.4 - Post-Generation Output Guardrails]
def validate_llm_output(output_text: str) -> dict:
    """
    Validates that model output does not leak secrets or adopt adversarial personas.
    """
    # [TODO: Inspect output for API keys, secret leakage, or jailbreak acknowledgment]
    pass


---
## 🎯 Part 4: End-to-End Secure Agent & Executable Test Suite (Core Skill 3)
**Weight: 30 Points | Official Locator: Tasks 4.1 & 4.2**

### 📋 Task 4.1: Production `SecureAnthropicAgent` Class
Integrate all guardrail components into an orchestrating agent class:
```
Input -> Base64 / Obfuscation Sanitizer -> Threat Heuristics -> XML Delimiters -> Anthropic Claude -> Output Validation
```

### 📋 Task 4.2: Automated Executable Unit Test Suite
Write unit test functions using `assert` statements that verify:
1. `test_benign_query_success()`: Legitimate queries succeed (`status == 'SUCCESS'`).
2. `test_direct_prompt_injection_blocked()`: Injections are intercepted (`status == 'BLOCKED'`).
3. `test_dan_jailbreak_blocked()`: DAN exploits are intercepted (`status == 'BLOCKED'`).
4. `test_persona_roleplay_blocked()`: Role-playing attacks are intercepted (`status == 'BLOCKED'`).
5. `test_base64_obfuscation_blocked()`: Base64 attacks are decoded and intercepted (`status == 'BLOCKED'`).


In [ ]:
# ==============================================================================
# SOLUTION CELL: Part 4 - End-to-End Secure Agent & Automated Test Suite
# Candidate ID: UST-CAND-001
# ==============================================================================

class SecureAnthropicAgent:
    """
    Production-grade Anthropic Agent with Multi-Layer Security Guardrails.
    """
    def __init__(self, system_mission: str = "You are a secure UST Global enterprise operations assistant."):
        self.system_mission = system_mission
        self.audit_log = []

    def execute(self, user_input: str) -> dict:
        # [TODO: Implement complete orchestrating pipeline]
        pass

# [TODO: Task 4.2 - Implement Automated Unit Tests with assert statements]
def run_security_test_suite():
    agent = SecureAnthropicAgent()
    print("[*] Running Automated Security Verification Test Suite...")
    
    # [TODO: Add executable assertions verifying benign queries and all 4 attack vectors]
    # assert ...
    print("[✓] All Security Guardrail Assertions Passed!")

# Execute test suite
# run_security_test_suite()


---
## 🧭 Part 5: VECTOR Architectural Reflection & Self-Evaluation
**Weight: 40% of Weekly Score | Fixed Role Target: V4-E4-C4-T4-O5-R4**

Evaluate your implementation against the official 6 VECTOR dimensions. Document your rationale and architectural evidence below:

### 1. Vision Clarity [V] (Target Level 4)
- **Architectural Judgment**: [TODO: Contrast naive LLM passthrough with your multi-layer security architecture. How does your design guarantee safe enterprise execution?]

### 2. Edge Judgment [E] (Target Level 4)
- **Decision Window & Logging**: [TODO: How does your agent make decisive block/sanitize decisions before API execution? Where are audit decisions captured?]

### 3. Context Fluency [C] (Target Level 4)
- **Multi-Encoding & Stream Inspection**: [TODO: How does your pipeline cross-reference raw user text, decoded Base64 payloads, and model system boundaries?]

### 4. Trust Architecture [T] (Target Level 4)
- **Zero-Trust & Delimiter Isolation**: [TODO: Explain how XML boundary tags (<user_input>) prevent user inputs from breaking into the instruction plane.]

### 5. Orchestration Intelligence [O] (Target Level 5)
- **Pipeline Decomposition**: [TODO: Outline the exact operational sequence from pre-flight sanitization to post-generation verification.]

### 6. Range [R] (Target Level 4)
- **Enterprise & Program Alignment**: [TODO: How does this prompt security guardrail layer connect to production SOC2, compliance, rate limiting, and observability in subsequent weeks?]
